# Plant disease ablation study — GPU runner

Runs the A0–A4 ablation ladder plus the MobileNetV2 reference (B0).

Works on **Kaggle** or **Colab**, detected automatically. The project sources are
written onto the runtime by this notebook, so nothing needs to be cloned or uploaded.

**On Kaggle** (recommended — 30 GPU hours per week, and no dataset download):
add the dataset via *Add Input* → search `new-plant-diseases-dataset`, set
*Accelerator* to a GPU under the notebook options, then *Save & Run All*, which
runs in the background and survives closing the browser.

**On Colab:** connect a GPU runtime, then *Run All*. You will be asked for Kaggle
credentials so the dataset can be downloaded.

Every cell is safe to re-run: finished arms are detected and skipped, so a dropped
runtime costs only the arm that was in flight.

## 1. Confirm the GPU is attached

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| GPUs:', gpus)
assert gpus, 'No GPU attached. Pick a GPU runtime in the kernel picker, then rerun.'

## 2. Write the project files

Creates the working directory and drops in the model builders, data pipeline,
training protocol, and analysis scripts.

In [ ]:
import os, pathlib

ON_KAGGLE = pathlib.Path('/kaggle/input').exists()
WORK = pathlib.Path('/kaggle/working/plant-disease' if ON_KAGGLE else '/content/plant-disease')
for sub in ('src', 'scripts', 'data'):
    (WORK / sub).mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print('environment:', 'Kaggle' if ON_KAGGLE else 'Colab')
print('working directory:', os.getcwd())

In [ ]:
%%writefile src/custom_cnn.py
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Rescaling, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_custom_cnn(input_shape=(224, 224, 3), num_classes=38):
    model = Sequential([
        Input(shape=input_shape),
        Rescaling(1. / 255),

        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

if __name__ == "__main__":
    model = build_custom_cnn()
    model.summary()


In [ ]:
%%writefile src/transfer_mobilenet.py
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Rescaling, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

def build_mobilenet_model(input_shape=(224, 224, 3), num_classes=38):
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    inputs = Input(shape=input_shape)
    x = Rescaling(scale=1. / 127.5, offset=-1)(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=predictions)

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

if __name__ == "__main__":
    model = build_mobilenet_model()
    model.summary()


In [ ]:
%%writefile src/models.py
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Activation, Add, BatchNormalization, Conv2D, Dense, DepthwiseConv2D,
    Dropout, GlobalAveragePooling2D, MaxPooling2D, ReLU, Rescaling, SeparableConv2D,
)

from custom_cnn import build_custom_cnn
from transfer_mobilenet import build_mobilenet_model

CHANNELS = (32, 64, 128)
DENSE_UNITS = 128
DROPOUT = 0.5


def _compile(model):
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def _head(x, num_classes):
    x = GlobalAveragePooling2D()(x)
    x = Dense(DENSE_UNITS, activation="relu")(x)
    x = Dropout(DROPOUT)(x)
    return Dense(num_classes, activation="softmax")(x)


def build_a0(input_shape=(224, 224, 3), num_classes=38):
    return build_custom_cnn(input_shape=input_shape, num_classes=num_classes)


def build_a1(input_shape=(224, 224, 3), num_classes=38):
    inputs = Input(shape=input_shape)
    x = Rescaling(1. / 255)(inputs)
    for filters in CHANNELS:
        x = Conv2D(filters, (3, 3), activation="relu")(x)
        x = MaxPooling2D((2, 2))(x)
    return _compile(Model(inputs, _head(x, num_classes), name="A1_gap"))


def build_a2(input_shape=(224, 224, 3), num_classes=38):
    inputs = Input(shape=input_shape)
    x = Rescaling(1. / 255)(inputs)
    for filters in CHANNELS:
        x = Conv2D(filters, (3, 3), use_bias=False)(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)
        x = MaxPooling2D((2, 2))(x)
    return _compile(Model(inputs, _head(x, num_classes), name="A2_batchnorm"))


def build_a3(input_shape=(224, 224, 3), num_classes=38):
    inputs = Input(shape=input_shape)
    x = Rescaling(1. / 255)(inputs)

    x = Conv2D(CHANNELS[0], (3, 3), use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = MaxPooling2D((2, 2))(x)

    for filters in CHANNELS[1:]:
        x = SeparableConv2D(filters, (3, 3), use_bias=False)(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)
        x = MaxPooling2D((2, 2))(x)

    return _compile(Model(inputs, _head(x, num_classes), name="A3_separable"))


def _inverted_residual(x, filters, expansion=4, stride=1):
    in_channels = x.shape[-1]
    y = Conv2D(in_channels * expansion, (1, 1), padding="same", use_bias=False)(x)
    y = BatchNormalization()(y)
    y = ReLU(max_value=6.0)(y)

    y = DepthwiseConv2D((3, 3), strides=stride, padding="same", use_bias=False)(y)
    y = BatchNormalization()(y)
    y = ReLU(max_value=6.0)(y)

    y = Conv2D(filters, (1, 1), padding="same", use_bias=False)(y)
    y = BatchNormalization()(y)

    if stride == 1 and in_channels == filters:
        y = Add()([x, y])
    return y


def build_a4(input_shape=(224, 224, 3), num_classes=38, expansion=4):
    inputs = Input(shape=input_shape)
    x = Rescaling(1. / 255)(inputs)

    x = Conv2D(CHANNELS[0], (3, 3), strides=2, padding="same", use_bias=False)(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6.0)(x)

    for stage, filters in enumerate(CHANNELS):
        x = _inverted_residual(x, filters, expansion=expansion,
                               stride=1 if stage == 0 else 2)
        x = _inverted_residual(x, filters, expansion=expansion, stride=1)

    return _compile(Model(inputs, _head(x, num_classes), name="A4_inverted_residual"))


def build_b0(input_shape=(224, 224, 3), num_classes=38):
    return build_mobilenet_model(input_shape=input_shape, num_classes=num_classes)


ARMS = {
    "A0": ("original custom CNN", build_a0),
    "A1": ("A0 + global average pooling head", build_a1),
    "A2": ("A1 + batch normalization", build_a2),
    "A3": ("A2 + depthwise separable convolutions", build_a3),
    "A4": ("A3 + inverted residual blocks", build_a4),
    "B0": ("MobileNetV2 transfer learning (reference)", build_b0),
}


def build(arm, input_shape=(224, 224, 3), num_classes=38):
    if arm not in ARMS:
        raise KeyError(f"unknown arm {arm!r}; expected one of {sorted(ARMS)}")
    return ARMS[arm][1](input_shape=input_shape, num_classes=num_classes)


if __name__ == "__main__":
    for name in sorted(ARMS):
        model = build(name, input_shape=(128, 128, 3))
        total = model.count_params()
        trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
        print(f"{name}  {ARMS[name][0]:<45} total={total:>12,}  trainable={trainable:>12,}")


In [ ]:
%%writefile src/data.py
import csv
import json
import random
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom

ROOT = Path(__file__).resolve().parents[1]
SPLIT_DIR = ROOT / "data" / "splits"

AUTOTUNE = tf.data.AUTOTUNE


def load_classes():
    with open(SPLIT_DIR / "split_meta.json", encoding="utf-8") as f:
        return json.load(f)["classes"]


def _read_manifest(split, classes):
    index = {name: i for i, name in enumerate(classes)}
    paths, labels = [], []
    with open(SPLIT_DIR / f"{split}.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            paths.append(str(ROOT / row["filepath"]))
            labels.append(index[row["class"]])
    return paths, labels


def _decode(path, label, img_size, num_classes):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, img_size, method="bilinear")
    return tf.cast(image, tf.uint8), tf.one_hot(label, num_classes)


def _cache_bytes(count, img_size):
    return count * img_size[0] * img_size[1] * 3


CACHE_LIMIT_BYTES = 3 * 1024 ** 3


def build_augmenter(seed):
    return tf.keras.Sequential([
        RandomFlip("horizontal", seed=seed),
        RandomRotation(0.1, seed=seed),
        RandomZoom(0.1, seed=seed),
    ], name="augmentation")


def make_dataset(split, img_size, batch_size, seed, classes=None, augment=False, shuffle=False):
    classes = classes or load_classes()
    paths, labels = _read_manifest(split, classes)
    num_classes = len(classes)

    if shuffle:
        order = list(range(len(paths)))
        random.Random(seed).shuffle(order)
        paths = [paths[i] for i in order]
        labels = [labels[i] for i in order]

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: _decode(p, l, img_size, num_classes), num_parallel_calls=AUTOTUNE)

    if _cache_bytes(len(paths), img_size) <= CACHE_LIMIT_BYTES:
        ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(min(len(paths), 4096), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if augment:
        augmenter = build_augmenter(seed)
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE), len(paths)


def true_labels(split, classes=None):
    classes = classes or load_classes()
    _, labels = _read_manifest(split, classes)
    return labels


In [ ]:
%%writefile scripts/make_splits.py
import argparse
import csv
import json
import random
import re
from collections import defaultdict
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data" / "archive" / "New Plant Diseases Dataset(Augmented)" / "New Plant Diseases Dataset(Augmented)"
OUT_DIR = ROOT / "data" / "splits"

AUG_SUFFIX = re.compile(r"_(\d+deg|flipTB|flipLR|newPixel\d*|new\w+|\w*[Nn]oise\w*)$")


def source_id(filename):
    return filename.split("___")[0]


def is_augmented(stem):
    return AUG_SUFFIX.search(stem) is not None


def scan():
    groups = defaultdict(list)
    group_class = {}

    for split_dir in ("train", "valid"):
        for class_dir in sorted((DATA_DIR / split_dir).iterdir()):
            if not class_dir.is_dir():
                continue
            for path in sorted(class_dir.iterdir()):
                uid = source_id(path.name)
                prev = group_class.setdefault(uid, class_dir.name)
                if prev != class_dir.name:
                    raise ValueError(
                        f"source image {uid} appears under two classes: {prev} / {class_dir.name}"
                    )
                groups[uid].append(path)

    return groups, group_class


def assign(groups, group_class, val_frac, test_frac, seed):
    by_class = defaultdict(list)
    for uid in groups:
        by_class[group_class[uid]].append(uid)

    rng = random.Random(seed)
    assignment = {}
    for class_name in sorted(by_class):
        uids = sorted(by_class[class_name])
        rng.shuffle(uids)
        n = len(uids)
        n_test = max(1, round(n * test_frac))
        n_val = max(1, round(n * val_frac))
        for uid in uids[:n_test]:
            assignment[uid] = "test"
        for uid in uids[n_test:n_test + n_val]:
            assignment[uid] = "val"
        for uid in uids[n_test + n_val:]:
            assignment[uid] = "train"
    return assignment


def build_rows(groups, group_class, assignment, cap_per_class, seed):
    rows = {"train": [], "val": [], "test": []}
    for uid, paths in groups.items():
        split = assignment[uid]
        for path in paths:
            augmented = is_augmented(path.stem)
            if split != "train" and augmented:
                continue
            rows[split].append({
                "filepath": path.relative_to(ROOT).as_posix(),
                "class": group_class[uid],
                "source_id": uid,
                "augmented": int(augmented),
            })

    if cap_per_class:
        rng = random.Random(seed + 1)
        capped = []
        by_class = defaultdict(list)
        for row in rows["train"]:
            by_class[row["class"]].append(row)
        for class_name in sorted(by_class):
            items = sorted(by_class[class_name], key=lambda r: r["filepath"])
            rng.shuffle(items)
            capped.extend(items[:cap_per_class])
        rows["train"] = capped

    for split in rows:
        rows[split].sort(key=lambda r: r["filepath"])
    return rows


def verify(rows):
    ids = {s: {r["source_id"] for r in rows[s]} for s in rows}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = ids[a] & ids[b]
        if overlap:
            raise AssertionError(f"{len(overlap)} source images shared between {a} and {b}")
    for split in ("val", "test"):
        if any(r["augmented"] for r in rows[split]):
            raise AssertionError(f"{split} contains augmented images")
    print("leakage check passed: no source image is shared across splits")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--val-frac", type=float, default=0.10)
    ap.add_argument("--test-frac", type=float, default=0.20)
    ap.add_argument("--cap-per-class", type=int, default=300,
                    help="max training files per class (reduced CPU protocol); 0 disables")
    args = ap.parse_args()

    groups, group_class = scan()
    print(f"scanned {sum(len(v) for v in groups.values()):,} files "
          f"from {len(groups):,} source photographs")

    assignment = assign(groups, group_class, args.val_frac, args.test_frac, args.seed)
    rows = build_rows(groups, group_class, assignment, args.cap_per_class, args.seed)
    verify(rows)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for split, items in rows.items():
        with open(OUT_DIR / f"{split}.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["filepath", "class", "source_id", "augmented"])
            w.writeheader()
            w.writerows(items)

    classes = sorted({r["class"] for r in rows["test"]})
    meta = {
        "seed": args.seed,
        "val_frac": args.val_frac,
        "test_frac": args.test_frac,
        "cap_per_class": args.cap_per_class,
        "num_classes": len(classes),
        "classes": classes,
        "counts": {s: len(v) for s, v in rows.items()},
        "source_photographs": {
            s: len({r["source_id"] for r in rows[s]}) for s in rows
        },
        "note": "grouped by source photograph; val/test are non-augmented originals only",
    }
    with open(OUT_DIR / "split_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print()
    for split in ("train", "val", "test"):
        n_files = len(rows[split])
        n_src = meta["source_photographs"][split]
        print(f"{split:<6} {n_files:>7,} files   {n_src:>7,} source photographs")
    print(f"\nmanifests written to {OUT_DIR}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/benchmark.py
import argparse
import statistics
import tempfile
import time
from pathlib import Path

import numpy as np
import tensorflow as tf

WARMUP = 20
RUNS = 100
THROUGHPUT_BATCH = 32


def _flops(model, input_shape):
    try:
        from tensorflow.python.profiler.model_analyzer import profile
        from tensorflow.python.profiler.option_builder import ProfileOptionBuilder

        forward = tf.function(lambda x: model(x, training=False)).get_concrete_function(
            tf.TensorSpec([1, *input_shape], tf.float32)
        )
        opts = ProfileOptionBuilder.float_operation()
        opts["output"] = "none"
        return int(profile(forward.graph, options=opts).total_float_ops)
    except Exception:
        return None


def _serialized_mb(model):
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "model.h5"
        try:
            model.save(path)
        except Exception:
            path = Path(tmp) / "model.keras"
            model.save(path)
        return round(path.stat().st_size / (1024 * 1024), 3)


def measure(model, input_shape, runs=RUNS, warmup=WARMUP):
    total = model.count_params()
    trainable = int(sum(tf.keras.backend.count_params(w) for w in model.trainable_weights))

    with tf.device("/CPU:0"):
        single = tf.constant(np.random.rand(1, *input_shape).astype("float32") * 255)
        for _ in range(warmup):
            model(single, training=False)

        timings = []
        for _ in range(runs):
            start = time.perf_counter()
            model(single, training=False)
            timings.append((time.perf_counter() - start) * 1000)
        timings.sort()

        batch = tf.constant(np.random.rand(THROUGHPUT_BATCH, *input_shape).astype("float32") * 255)
        for _ in range(3):
            model(batch, training=False)
        start = time.perf_counter()
        for _ in range(10):
            model(batch, training=False)
        elapsed = time.perf_counter() - start

    return {
        "total_params": int(total),
        "trainable_params": trainable,
        "frozen_params": int(total) - trainable,
        "model_size_mb": _serialized_mb(model),
        "flops": _flops(model, input_shape),
        "latency_ms_median": round(statistics.median(timings), 3),
        "latency_ms_p90": round(timings[int(0.9 * len(timings)) - 1], 3),
        "throughput_img_per_s": round(THROUGHPUT_BATCH * 10 / elapsed, 1),
        "latency_runs": runs,
        "latency_device": "CPU",
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model_path")
    ap.add_argument("--img-size", type=int, default=128)
    args = ap.parse_args()

    model = tf.keras.models.load_model(args.model_path)
    stats = measure(model, (args.img_size, args.img_size, 3))
    width = max(len(k) for k in stats)
    for key, value in stats.items():
        print(f"{key:<{width}}  {value}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/train_ablation.py
import argparse
import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

import data as data_lib
from benchmark import measure
from models import ARMS, build

RESULTS_DIR = ROOT / "results"


class EpochTimer(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(round(time.perf_counter() - self._start, 2))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arm", required=True, choices=sorted(ARMS))
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--img-size", type=int, default=128)
    ap.add_argument("--batch-size", type=int, default=32)
    args = ap.parse_args()

    tf.keras.utils.set_random_seed(args.seed)

    classes = data_lib.load_classes()
    img_size = (args.img_size, args.img_size)
    input_shape = (args.img_size, args.img_size, 3)

    train_ds, n_train = data_lib.make_dataset(
        "train", img_size, args.batch_size, args.seed, classes, augment=True, shuffle=True)
    val_ds, n_val = data_lib.make_dataset(
        "val", img_size, args.batch_size, args.seed, classes)
    test_ds, n_test = data_lib.make_dataset(
        "test", img_size, args.batch_size, args.seed, classes)

    run_dir = RESULTS_DIR / f"{args.arm}_seed{args.seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    weights_path = run_dir / "best.weights.h5"

    model = build(args.arm, input_shape=input_shape, num_classes=len(classes))
    print(f"\n{'=' * 64}\n{args.arm}: {ARMS[args.arm][0]}  (seed {args.seed})\n{'=' * 64}")
    print(f"train {n_train:,} | val {n_val:,} | test {n_test:,} | {len(classes)} classes")

    timer = EpochTimer()
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(weights_path), monitor="val_accuracy", mode="max",
            save_best_only=True, save_weights_only=True, verbose=0),
        tf.keras.callbacks.CSVLogger(str(run_dir / "history.csv")),
        timer,
    ]

    start = time.perf_counter()
    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=args.epochs, callbacks=callbacks, verbose=2)
    train_seconds = time.perf_counter() - start

    val_acc = history.history["val_accuracy"]
    best_epoch = int(np.argmax(val_acc)) + 1
    model.load_weights(weights_path)

    probs = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
    assert len(y_true) == len(y_pred), f"{len(y_true)} labels vs {len(y_pred)} predictions"

    report = classification_report(
        y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
    efficiency = measure(model, input_shape)

    results = {
        "arm": args.arm,
        "description": ARMS[args.arm][0],
        "seed": args.seed,
        "protocol": {
            "img_size": args.img_size,
            "batch_size": args.batch_size,
            "epochs": args.epochs,
            "n_train": n_train,
            "n_val": n_val,
            "n_test": n_test,
            "selection": "best val_accuracy epoch, no early stopping",
        },
        "training": {
            "total_seconds": round(train_seconds, 1),
            "seconds_per_epoch_median": round(float(np.median(timer.times)), 1),
            "epoch_seconds": timer.times,
            "best_epoch": best_epoch,
            "best_val_accuracy": round(float(max(val_acc)), 5),
            "final_train_accuracy": round(float(history.history["accuracy"][-1]), 5),
            "final_val_accuracy": round(float(val_acc[-1]), 5),
            "generalization_gap": round(
                float(history.history["accuracy"][best_epoch - 1] - val_acc[best_epoch - 1]), 5),
        },
        "test": {
            "accuracy": round(float(accuracy_score(y_true, y_pred)), 5),
            "macro_precision": round(report["macro avg"]["precision"], 5),
            "macro_recall": round(report["macro avg"]["recall"], 5),
            "macro_f1": round(report["macro avg"]["f1-score"], 5),
            "weighted_f1": round(report["weighted avg"]["f1-score"], 5),
        },
        "efficiency": efficiency,
        "per_class_f1": {c: round(report[c]["f1-score"], 5) for c in classes},
    }

    with open(run_dir / "results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    np.savez_compressed(run_dir / "test_predictions.npz", y_true=y_true, y_pred=y_pred)

    t = results["test"]
    e = results["efficiency"]
    print(f"\ntest accuracy {t['accuracy']:.4f} | macro-F1 {t['macro_f1']:.4f}")
    print(f"params {e['total_params']:,} | {e['model_size_mb']} MB | "
          f"{e['latency_ms_median']} ms/image")
    print(f"results -> {run_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_ablation.py
import argparse
import os
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
RESULTS_DIR = ROOT / "results"
DEFAULT_ARMS = ["A0", "A1", "A2", "A3", "A4", "B0"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arms", nargs="+", default=DEFAULT_ARMS)
    ap.add_argument("--seeds", nargs="+", type=int, default=[42])
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--img-size", type=int, default=128)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--force", action="store_true")
    args = ap.parse_args()

    jobs = [(a, s) for s in args.seeds for a in args.arms]
    print(f"{len(jobs)} run(s) queued: {args.arms} x seeds {args.seeds}\n")

    started = time.perf_counter()
    for i, (arm, seed) in enumerate(jobs, 1):
        out = RESULTS_DIR / f"{arm}_seed{seed}" / "results.json"
        if out.exists() and not args.force:
            print(f"[{i}/{len(jobs)}] {arm} seed {seed}: already complete, skipping")
            continue

        print(f"[{i}/{len(jobs)}] {arm} seed {seed}: starting", flush=True)
        t0 = time.perf_counter()
        env = {**os.environ, "PYTHONUNBUFFERED": "1"}
        proc = subprocess.run([
            sys.executable, "-u", str(ROOT / "scripts" / "train_ablation.py"),
            "--arm", arm, "--seed", str(seed),
            "--epochs", str(args.epochs),
            "--img-size", str(args.img_size),
            "--batch-size", str(args.batch_size),
        ], env=env)
        mins = (time.perf_counter() - t0) / 60
        status = "done" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
        print(f"[{i}/{len(jobs)}] {arm} seed {seed}: {status} in {mins:.1f} min\n", flush=True)

    print(f"sweep finished in {(time.perf_counter() - started) / 60:.1f} min")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/analyze_ablation.py
import argparse
import csv
import json
from pathlib import Path

import numpy as np
from scipy.stats import binomtest

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parents[1]
FIG_DIR = ROOT / "output"

ORDER = ["A0", "A1", "A2", "A3", "A4", "B0"]


def load_runs(seed, results_dir):
    runs = {}
    for path in sorted(results_dir.glob(f"*_seed{seed}/results.json")):
        with open(path, encoding="utf-8") as f:
            r = json.load(f)
        runs[r["arm"]] = r
        preds = np.load(path.parent / "test_predictions.npz")
        r["_y_true"] = preds["y_true"]
        r["_y_pred"] = preds["y_pred"]
    return {a: runs[a] for a in ORDER if a in runs}


def mcnemar(a, b):
    correct_a = a["_y_pred"] == a["_y_true"]
    correct_b = b["_y_pred"] == b["_y_true"]
    only_a = int(np.sum(correct_a & ~correct_b))
    only_b = int(np.sum(~correct_a & correct_b))
    n = only_a + only_b
    p = binomtest(only_b, n, 0.5).pvalue if n else 1.0
    return only_a, only_b, p


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--results-dir", default="output",
                    help="directory holding the per-arm run folders")
    args = ap.parse_args()

    results_dir = Path(args.results_dir)
    if not results_dir.is_absolute():
        results_dir = ROOT / results_dir

    runs = load_runs(args.seed, results_dir)
    if not runs:
        raise SystemExit(f"no results found for seed {args.seed} in {results_dir}")

    baseline = runs.get("A0")
    rows = []
    for arm, r in runs.items():
        t, e, tr = r["test"], r["efficiency"], r["training"]
        row = {
            "arm": arm,
            "description": r["description"],
            "test_accuracy": t["accuracy"],
            "macro_f1": t["macro_f1"],
            "params": e["total_params"],
            "size_mb": e["model_size_mb"],
            "latency_ms": e["latency_ms_median"],
            "sec_per_epoch": tr["seconds_per_epoch_median"],
            "best_epoch": tr["best_epoch"],
            "gen_gap": tr["generalization_gap"],
        }
        if baseline and arm != "A0":
            row["d_macro_f1_vs_A0"] = round(t["macro_f1"] - baseline["test"]["macro_f1"], 5)
            row["param_ratio_vs_A0"] = round(
                e["total_params"] / baseline["efficiency"]["total_params"], 4)
            wins_a0, wins_arm, p = mcnemar(baseline, r)
            row["mcnemar_p_vs_A0"] = f"{p:.2e}"
            row["disagreements"] = f"{wins_a0}/{wins_arm}"
        rows.append(row)

    results_dir.mkdir(parents=True, exist_ok=True)
    with open(results_dir / f"summary_seed{args.seed}.csv", "w", newline="", encoding="utf-8") as f:
        keys = sorted({k for r in rows for k in r})
        w = csv.DictWriter(f, fieldnames=["arm", "description"] + [k for k in keys if k not in ("arm", "description")])
        w.writeheader()
        w.writerows(rows)

    hdr = f"{'arm':<4} {'macro-F1':>9} {'acc':>7} {'params':>11} {'MB':>7} {'ms':>7} {'s/ep':>7} {'gap':>7}"
    print("\n" + hdr)
    print("-" * len(hdr))
    for r in rows:
        print(f"{r['arm']:<4} {r['macro_f1']:>9.4f} {r['test_accuracy']:>7.4f} "
              f"{r['params']:>11,} {r['size_mb']:>7.2f} {r['latency_ms']:>7.2f} "
              f"{r['sec_per_epoch']:>7.0f} {r['gen_gap']:>7.3f}")

    if baseline:
        print("\nvs A0 (McNemar exact, discordant pairs A0-only/arm-only):")
        for r in rows:
            if "mcnemar_p_vs_A0" in r:
                print(f"  {r['arm']}: macro-F1 {r['d_macro_f1_vs_A0']:+.4f}  "
                      f"params x{r['param_ratio_vs_A0']:.3f}  "
                      f"p={r['mcnemar_p_vs_A0']}  ({r['disagreements']})")

    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    markers = ["o", "s", "^", "D", "v", "*"]
    for ax, xkey, xlabel, logx in (
        (axes[0], "params", "Parameters", True),
        (axes[1], "latency_ms", "CPU latency per image (ms)", False),
    ):
        for i, r in enumerate(rows):
            face = "black" if r["arm"].startswith("A") else "white"
            ax.scatter(r[xkey], r["macro_f1"], marker=markers[i % len(markers)],
                       s=110, facecolors=face, edgecolors="black", zorder=3,
                       label=f"{r['arm']}")
            ax.annotate(r["arm"], (r[xkey], r["macro_f1"]),
                        textcoords="offset points", xytext=(7, -3), fontsize=9)
        if logx:
            ax.set_xscale("log")
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Macro F1 (test)")
        ax.grid(alpha=0.3, zorder=0)
    axes[0].legend(loc="lower right", fontsize=8, frameon=True)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "ablation_tradeoff.png", dpi=180)
    plt.close()

    print(f"\nsummary -> {results_dir / f'summary_seed{args.seed}.csv'}")
    print(f"figure  -> {FIG_DIR / 'ablation_tradeoff.png'}")


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, sys

missing = [f for f in ['src/custom_cnn.py', 'src/transfer_mobilenet.py', 'src/models.py', 'src/data.py', 'scripts/make_splits.py', 'scripts/benchmark.py', 'scripts/train_ablation.py', 'scripts/run_ablation.py', 'scripts/analyze_ablation.py'] if not pathlib.Path(f).exists()]
assert not missing, f'files failed to write: {missing}'
print(len(['src/custom_cnn.py', 'src/transfer_mobilenet.py', 'src/models.py', 'src/data.py', 'scripts/make_splits.py', 'scripts/benchmark.py', 'scripts/train_ablation.py', 'scripts/run_ablation.py', 'scripts/analyze_ablation.py']), 'source files written')

## 3. Dataset

On Kaggle the attached dataset is linked in place, which costs nothing. On Colab it
is downloaded (~2.7 GB) straight onto the runtime, so nothing uploads from your
machine; credentials come from `KAGGLE_USERNAME` / `KAGGLE_KEY`, an existing
`~/.kaggle/kaggle.json`, or a hidden prompt.

For Colab, get a token at kaggle.com → Settings → API → *Create New Token*, and copy
the two values out of the downloaded `kaggle.json`. Accept the dataset's terms on its
Kaggle page first, or the download returns 403.

In [ ]:
import os, pathlib

DATA_ROOT = pathlib.Path('data/archive/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)')

def locate(base):
    for train in pathlib.Path(base).rglob('train'):
        if train.is_dir() and sum(1 for c in train.iterdir() if c.is_dir()) > 30:
            return train.parent
    return None

if ON_KAGGLE and not (DATA_ROOT / 'train').exists():
    source = locate('/kaggle/input')
    assert source, 'Attach the dataset first: Add Input -> new-plant-diseases-dataset'
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if not DATA_ROOT.exists():
        os.symlink(source, DATA_ROOT)
    print('linked attached dataset from', source)

HAVE_DATA = (DATA_ROOT / 'train').exists()
print('dataset ready, no download needed' if HAVE_DATA else 'dataset missing, will download')

In [ ]:
import os, json, getpass, pathlib

if not HAVE_DATA:
    cred = pathlib.Path.home() / '.kaggle' / 'kaggle.json'
    if cred.exists():
        blob = json.loads(cred.read_text())
        os.environ['KAGGLE_USERNAME'] = blob['username']
        os.environ['KAGGLE_KEY'] = blob['key']
    elif not os.environ.get('KAGGLE_KEY'):
        os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ').strip()
        os.environ['KAGGLE_KEY'] = getpass.getpass('Kaggle key (hidden): ').strip()
    print('credentials ready')

In [ ]:
if not HAVE_DATA:
    !pip install -q kaggle
    !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /tmp/pd --force
    !mkdir -p data/archive
    !unzip -q -o /tmp/pd/new-plant-diseases-dataset.zip -d data/archive

In [ ]:
import pathlib

if not (DATA_ROOT / 'train').exists():
    for candidate in pathlib.Path('data/archive').rglob('train'):
        if candidate.is_dir() and any(candidate.iterdir()):
            DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
            candidate.parent.rename(DATA_ROOT)
            break

assert (DATA_ROOT / 'train').exists(), f'could not locate train/ under {DATA_ROOT}'
print('train classes:', len(list((DATA_ROOT / 'train').iterdir())))
print('valid classes:', len(list((DATA_ROOT / 'valid').iterdir())))

## 4. Build leakage-free splits

Groups every file by the source photograph encoded in its filename, so no photograph
can appear in more than one split. Validation and test keep non-augmented originals
only.

This is what makes the study valid: 62.7% of the dataset's own `valid/` folder are
rotations and flips of images that also sit in `train/`.

In [ ]:
!python scripts/make_splits.py --seed 42 --cap-per-class 600

## 5. Calibrate epoch cost

One short run to measure this GPU's real speed, so the sweep can be sized before
committing hours to it.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 40

!python scripts/train_ablation.py --arm A2 --seed 99 --epochs 1 --img-size {IMG_SIZE} --batch-size {BATCH_SIZE}

In [ ]:
import json, shutil, pathlib

probe = pathlib.Path('results/A2_seed99/results.json')
sec = json.load(open(probe))['training']['seconds_per_epoch_median']
shutil.rmtree(probe.parent)

print(f'{sec:.0f}s per epoch')
print(f'estimated sweep: {sec * EPOCHS * 6 / 3600:.1f} h for 6 arms x {EPOCHS} epochs')
print('If that is too long, lower EPOCHS or IMG_SIZE above and rerun this section.')

## 6. Run the ablation ladder

Every arm gets an identical budget and selection rule. Rerun this cell after a
disconnect and completed arms are skipped.

Epochs are set high enough to reach convergence, which the CPU protocol could not
afford; under-trained arms were the main flaw of the earlier attempt.

In [ ]:
!python scripts/run_ablation.py --seeds 42 --epochs {EPOCHS} --img-size {IMG_SIZE} --batch-size {BATCH_SIZE}

## 7. Analyse

Comparison table, McNemar exact tests against the A0 baseline, and a
black-and-white accuracy/efficiency trade-off figure.

In [ ]:
!python scripts/analyze_ablation.py --seed 42

In [ ]:
from IPython.display import Image, display

display(Image('output/ablation_tradeoff.png'))

## 8. Check convergence

If validation accuracy is still climbing at the final epoch then the budget was too
short, and the comparison is measuring convergence speed rather than architecture.
Anything flagged `STILL IMPROVING` cannot be trusted yet.

In [ ]:
import csv, pathlib

for run in sorted(pathlib.Path('results').glob('*_seed42')):
    rows = list(csv.DictReader(open(run / 'history.csv')))
    if not rows:
        continue
    val = [float(r['val_accuracy']) for r in rows]
    best = max(val)
    tail_gain = best - max(val[:max(1, len(val) - 5)])
    flag = 'STILL IMPROVING' if tail_gain > 0.005 else 'converged'
    print(f'{run.name:<12} best val {best:.4f} at epoch {val.index(best) + 1:>3} | last-5 gain {tail_gain:+.4f} -> {flag}')

## 9. Collect results

Zips the results and prints a compact summary.

On Kaggle the archive is written into `/kaggle/working`, which is what a committed run
saves as its output; anything written elsewhere is discarded when the session ends.

In [ ]:
import json, pathlib, shutil

dest = '/kaggle/working/ablation_results' if ON_KAGGLE else '/tmp/ablation_results'
archive = shutil.make_archive(dest, 'zip', '.', 'results')
print('archive:', archive, f'({pathlib.Path(archive).stat().st_size / 1e6:.1f} MB)')

if ON_KAGGLE and DATA_ROOT.is_symlink():
    DATA_ROOT.unlink()

print()
for run in sorted(pathlib.Path('results').glob('*_seed42')):
    r = json.load(open(run / 'results.json'))
    t, e = r['test'], r['efficiency']
    print(f"{r['arm']:<3} macroF1={t['macro_f1']:.4f} acc={t['accuracy']:.4f} "
          f"params={e['total_params']:,} size={e['model_size_mb']}MB "
          f"lat={e['latency_ms_median']}ms")

try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print('\nAutomatic download unavailable:', type(exc).__name__)
    print('Download the archive path above from the file explorer.')